# Objective
# Feature Engineering

The objective of this notebook is to create meaningful features from the cleaned Google Play Store datasets.

Feature engineering helps transform raw variables into analytical features that better represent:
- App popularity
- User engagement
- Monetization strategy
- Developer influence
- App lifecycle
- Market performance

These features will support SQL analysis, Power BI dashboards, and future machine learning models.

## Sections in this Notebook

1. Import Libraries

2. Load Cleaned Data

3. Review Existing Features

4. App Popularity Features

5. User Engagement Features

6. Monetization Features

7. Developer Intelligence Features

8. App Lifecycle Features

9. Country-Level Features

10. Discovery Features

11. Feature Validation

12. Export Final Analytical Datasets

# 1. Load Data

In [2]:
import pandas as pd
import numpy as np

apps = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\apps_clean.csv")

country = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\country_clean.csv")

discovery = pd.read_csv(r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\cleaned\discovery_clean.csv")

# 2. Review Existing Features

In [3]:
apps.columns

Index(['app_id', 'title', 'description', 'summary', 'installs', 'min_installs',
       'max_installs', 'score', 'ratings', 'reviews', 'histogram', 'price',
       'free', 'currency', 'sale', 'developer', 'developer_id',
       'developer_email', 'developer_website', 'privacy_policy', 'genre',
       'genre_id', 'categories', 'icon', 'header_image', 'screenshots',
       'content_rating', 'ad_supported', 'contains_ads', 'in_app_purchases',
       'updated', 'version', 'scraped_at', 'estimated_installs', 'update_year',
       'update_month', 'days_since_update', 'update_missing'],
      dtype='object')

# 3. App Popularity Features

### Feature 1: Log Installs

Why?

Installs are highly right-skewed.

A few apps have billions of installs.

Instead of:

`1,000,000,000`

use:

`log(1,000,000,000)`

In [4]:
apps["log_installs"] = np.log1p(
    apps["estimated_installs"]
)

### Log Transformation of Installs

The install distribution is highly right-skewed because a small number of applications have extremely high download counts.

A logarithmic transformation reduces the effect of extreme values and creates a more balanced feature for analysis and machine learning.

### Feature 2: Install Category

In [5]:
apps["install_category"] = pd.cut(
    apps["estimated_installs"],
    bins=[
        0,
        10000,
        100000,
        1000000,
        10000000,
        np.inf
    ],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

# 4. User Engagement Features

### Feature 3: Review Rate

Question:

How many users leave reviews after installing?

Formula:

`reviews / installs`

In [6]:
apps["review_rate"] = (
    apps["reviews"] /
    apps["estimated_installs"]
)

apps["review_rate"] = (
    apps["review_rate"]
    .replace([np.inf,-np.inf],np.nan)
    .fillna(0)
)

### Review Rate

Review count alone is influenced by app popularity.

Review rate measures user engagement by calculating the proportion of users who provide feedback after installation.

### Feature 4: Rating Strength

#####  Problem:

A 5-star rating from 10 users is not equal to 4.5 stars from 1 million users.


In [7]:
apps["rating_strength"] = (
    apps["score"] *
    np.log1p(apps["ratings"])
)

# 5. Monetization Features

### Feature 5: Paid App Indicator

Already have:

`free`

In [8]:
apps["paid_app"] = (
    ~apps["free"]
)

### Feature 6: Price Category

In [9]:
apps["price_category"] = pd.cut(
    apps["price"],
    bins=[
        -1,
        0,
        2,
        5,
        np.inf
    ],
    labels=[
        "Free",
        "Low Cost",
        "Medium Cost",
        "Premium"
    ]
)

# 6. Developer Intelligence Features

Very useful feature.

Question:

Does developer experience affect success?

Create developer app count:

In [10]:
developer_app_count = (
    apps.groupby("developer_id")
    ["app_id"]
    .count()
)

apps["developer_app_count"] = (
    apps["developer_id"]
    .map(developer_app_count)
)

### Developer Experience Feature

Developers publishing multiple applications may have more experience, resources, and established user bases.

The number of applications published by each developer is used as a proxy for developer experience.

# 7. App Lifecycle Features

We already created
`days_since_update`

Now, Update Recency Group

In [11]:
apps["update_frequency"] = pd.cut(
    apps["days_since_update"],
    bins=[
        0,
        30,
        180,
        365,
        np.inf
    ],
    labels=[
        "Recently Updated",
        "Updated Recently",
        "Old Update",
        "Very Old"
    ]
)

# 8. Country Features (From country dataset):

In [12]:
country_summary = (
    country.groupby("country")
    .agg(
        country_avg_installs=("min_installs","mean"),
        country_avg_rating=("score","mean"),
        country_total_reviews=("reviews","sum")
    )
)

# 9. Discovery Features

From discovery dataset:

Number of discovery appearances:

In [13]:
discovery_count = (
    discovery.groupby("app_id")
    .size()
)

apps["discovery_count"] = (
    apps["app_id"]
    .map(discovery_count)
    .fillna(0)
)

###  10. Validating Features

In [14]:
apps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11176 entries, 0 to 11175
Data columns (total 47 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   app_id               11176 non-null  object  
 1   title                11176 non-null  object  
 2   description          11176 non-null  object  
 3   summary              11176 non-null  object  
 4   installs             11176 non-null  object  
 5   min_installs         11176 non-null  float64 
 6   max_installs         11176 non-null  int64   
 7   score                11176 non-null  float64 
 8   ratings              11176 non-null  float64 
 9   reviews              11176 non-null  float64 
 10  histogram            11176 non-null  object  
 11  price                11176 non-null  float64 
 12  free                 11176 non-null  bool    
 13  currency             11176 non-null  object  
 14  sale                 11176 non-null  bool    
 15  developer          

In [15]:
apps.isnull().sum()

app_id                    0
title                     0
description               0
summary                   0
installs                  0
min_installs              0
max_installs              0
score                     0
ratings                   0
reviews                   0
histogram                 0
price                     0
free                      0
currency                  0
sale                      0
developer                 0
developer_id              0
developer_email           0
developer_website         0
privacy_policy            0
genre                     0
genre_id                  0
categories                0
icon                      0
header_image              0
screenshots               0
content_rating            0
ad_supported              0
contains_ads              0
in_app_purchases          0
updated                3394
version                   0
scraped_at                0
estimated_installs        0
update_year            3394
update_month        

These are not actually missing installs. They have:

estimated_installs = 0

The issue is our binning logic.

In [16]:
apps[apps["install_category"].isnull()][
    ["title","estimated_installs"]
]

,title,estimated_installs
7298,Budget Planner - Money Manager,0.0
7635,Meow Knights: Idle RPG,0.0
7683,Toy Survivor : The War,0.0
7688,Merge City - Travel & Story,0.0
7692,Mancala Game Online,0.0
7766,Kakatu: Pronunciation Coach,0.0
7849,Notee: AI Meeting Note Taker,0.0
7888,AI Skincare - Face Scanner,0.0
8170,AI Duplicate Photo Cleaner,0.0
8289,Warranty Tracker,0.0


In [17]:
apps["install_category"] = pd.cut(
    apps["estimated_installs"],
    bins=[
        -1,
        10000,
        100000,
        1000000,
        10000000,
        np.inf
    ],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [18]:
apps["install_category"].isnull().sum()

0

#### Fixed

#### Correction of Install Category Binning

During feature validation, 18 missing values were found in the `install_category` feature.

Investigation showed that these were not missing install values. They were applications with `estimated_installs = 0`.

The missing categories occurred because the initial binning range started from 0, while `pd.cut()` excludes the lowest boundary by default.

The binning logic was corrected by changing the lower boundary to `-1`, allowing applications with zero estimated installs to be categorized properly.

After correction, no missing values remained in the `install_category` feature.

# Saving Final Dataset

In [22]:
apps.to_csv(
    r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\processed\apps_processed.csv",
    index=False
)

country.to_csv(
    r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\processed\country_processed.csv",
    index=False
)

discovery.to_csv(
    r"C:\Users\cashk\OneDrive\Desktop\Projects\Google_Playstore_project\Playstore_Data\processed\discovery_processed.csv",
    index=False
)

# Feature Engineering Summary and Key Insights

Feature engineering was performed to transform cleaned application data into meaningful analytical variables that better represent app performance, user engagement, monetization strategy, developer influence, and market visibility.

The objective was not only to create additional columns but to convert raw attributes into features that can support deeper analysis, SQL business queries, Power BI dashboards, and future machine learning models.

## 1. App Popularity and Scale Features

- The original install distribution was highly right-skewed, with a small number of applications achieving extremely high download counts.
- A logarithmic transformation was applied to create `log_installs`, reducing the impact of extreme values and making app popularity easier to compare.
- `install_category` was created to group applications into popularity segments:
  - Very Low
  - Low
  - Medium
  - High
  - Very High

This allows easier comparison of app performance across business segments rather than relying only on raw install numbers.

## 2. User Engagement Features

- Install count alone does not represent user engagement, therefore additional engagement-based features were created.
- `review_rate` was introduced to measure the proportion of users leaving reviews relative to estimated installs.
- `rating_strength` was created by combining app rating score with the number of ratings, reducing the bias caused by applications having high ratings from very few users.

These features provide a more balanced understanding of app quality and user interaction.

## 3. Monetization Features

- Although the dataset already contained free/paid information, additional monetization features were created for analysis.
- `paid_app` provides a simplified indicator for monetization strategy.
- `price_category` groups applications into free, low-cost, medium-cost, and premium segments.

The price distribution was highly skewed, therefore categorical pricing groups provide better business interpretation than raw price values alone.

## 4. Developer Intelligence Features

- Developer influence was captured by creating `developer_app_count`.
- This feature represents the number of applications published by each developer and acts as a proxy for developer experience or publishing activity.

This enables analysis of whether developers with larger portfolios achieve better market performance.

## 5. App Lifecycle Features

- Update-related features were created to understand application maintenance behavior.
- Existing update information was transformed into:
    - `update_year`
    - `update_month`
    - `days_since_update`
    - `update_frequency`

- Missing update dates were intentionally preserved because the absence of update information may itself indicate lower maintenance activity.
- The `update_missing` indicator allows future analysis to distinguish between available and unavailable update history.

## 6. Discovery Visibility Features

- The discovery dataset was used to create `discovery_count`.
- This feature measures how frequently an application appeared across discovery signals such as search results and chart pages.

This provides additional context because app success is influenced not only by quality but also by visibility and discoverability.

## 7. Feature Validation

After feature creation:

- New features were checked for missing values.
- Unexpected missing values caused by feature creation logic were corrected.
- Existing missing information, such as unavailable update dates, was preserved instead of artificially filling values.

This ensured that engineered features maintained the original meaning of the data.


# Conclusion

Feature engineering converted the cleaned Google Play Store datasets into a more analytical structure by creating variables that represent different dimensions of app performance.

The major improvements introduced were:

- Popularity measurement through transformed install metrics and install segments.
- Better engagement measurement through review-based and rating-based features.
- Monetization analysis through pricing groups and paid/free indicators.
- Developer capability measurement through application portfolio size.
- Application lifecycle analysis through update behavior.
- Visibility measurement through discovery frequency.

These engineered features provide a stronger foundation for the next stages of the project. Instead of analyzing apps only through raw attributes, future SQL analysis and dashboards can evaluate applications based on performance segments, engagement quality, developer influence, market visibility, and lifecycle behavior.

The final featured datasets are now ready for:
- SQL-based business analysis,
- Power BI dashboard development,
- and potential machine learning modeling.